# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania: 2.64s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Godność osoby ludzkiej
2. Pan Wołodyjowski
3. Nie budźcie mnie
4. Szymborska. Kropki, przecinki, papierosy
5. Wspólnie. 20 lat kolektywu Sputnik Photos
6. Rodzeństwo (Ritter, Dene, Voss)
7. Męczeństwo i śmierć Marata
8. Gotyk w Karpatach
9. Fisz Emade Tworzywo w Studio
10. Nie ma jak w domu. Projekt fotograficzny Agaty Mendziuk

Czas wykonania (wątki): 0.73s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [5]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 1.60s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [6]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_FACTS = 20

def get_cat_fact(_=None):
    try:
        response = requests.get(CAT_API_URL, timeout=5)
        return response.json().get('fact')
    except Exception as e:
        return f"Błąd: {e}"

print(f"Pobieranie {NUM_FACTS} faktów sekwencyjnie...")
start_seq = time.time()
facts_seq = []
for _ in range(NUM_FACTS):
    facts_seq.append(get_cat_fact())
end_seq = time.time()
print(f"Sekwencyjnie: {end_seq - start_seq:.2f}s")

print(f"\nPobieranie {NUM_FACTS} faktów wielowątkowo...")
start_threads = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    facts_threads = list(executor.map(get_cat_fact, range(NUM_FACTS)))
end_threads = time.time()
print(f"Wielowątkowo: {end_threads - start_threads:.2f}s")

print(f"\nZysk czasowy: {((end_seq - start_seq) / (end_threads - start_threads)):.1f}x szybciej.")

Pobieranie 20 faktów sekwencyjnie...
Sekwencyjnie: 4.92s

Pobieranie 20 faktów wielowątkowo...
Wielowątkowo: 0.55s

Zysk czasowy: 9.0x szybciej.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [7]:
import queue
import threading
import time

def producer(q, limit):
    for i in range(1, limit + 1):
        print(f"[Producent] Generuję: {i}")
        q.put(i)
        time.sleep(0.1)
    q.put(None)
    q.put(None)

def consumer_even(q):
    while True:
        num = q.get()
        if num is None: break
        if num % 2 == 0:
            print(f"  [Konsument 1 - Parzyste] Przetwarzam: {num}")
        else:
            q.put(num)
            time.sleep(0.01)

def consumer_odd(q):
    while True:
        num = q.get()
        if num is None: break
        if num % 2 != 0:
            print(f"  [Konsument 2 - Nieparzyste] Przetwarzam: {num}")
        else:
            q.put(num)
            time.sleep(0.01)

q = queue.Queue(maxsize=10)
limit = 10

t1 = threading.Thread(target=producer, args=(q, limit))
t2 = threading.Thread(target=consumer_even, args=(q,))
t3 = threading.Thread(target=consumer_odd, args=(q,))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()
print("Przetwarzanie zakończone.")

[Producent] Generuję: 1
  [Konsument 2 - Nieparzyste] Przetwarzam: 1
[Producent] Generuję: 2
  [Konsument 1 - Parzyste] Przetwarzam: 2
[Producent] Generuję: 3
  [Konsument 2 - Nieparzyste] Przetwarzam: 3
[Producent] Generuję: 4
  [Konsument 1 - Parzyste] Przetwarzam: 4
[Producent] Generuję: 5
  [Konsument 2 - Nieparzyste] Przetwarzam: 5
[Producent] Generuję: 6
  [Konsument 1 - Parzyste] Przetwarzam: 6
[Producent] Generuję: 7
  [Konsument 2 - Nieparzyste] Przetwarzam: 7
[Producent] Generuję: 8
  [Konsument 1 - Parzyste] Przetwarzam: 8
[Producent] Generuję: 9
  [Konsument 2 - Nieparzyste] Przetwarzam: 9
[Producent] Generuję: 10
  [Konsument 1 - Parzyste] Przetwarzam: 10
Przetwarzanie zakończone.


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [11]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    numbers = range(1, 10001)
    
    cores = multiprocessing.cpu_count()
    print(f"Rozpoczynam obliczenia na {cores} procesach...")
    
    start = time.time()
    
    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)
    
    end = time.time()
    
    print(f"Obliczono sumy dla {len(results)} liczb.")
    print(f"Suma pierwszej liczby (przykład): {results[0]}")
    print(f"Czas wykonania (Multiprocessing): {end - start:.2f}s")

Rozpoczynam obliczenia na 12 procesach...
Obliczono sumy dla 10000 liczb.
Suma pierwszej liczby (przykład): 100
Czas wykonania (Multiprocessing): 0.29s
